In [12]:
import pandas as pd

df = pd.read_csv('.\study-w10p1\data.csv')
df

,Day,Hour,Minute,DHI,DNI,WS,RH,T,TARGET
0,102,0,0,0,0,1.4,57.76,0,0.000000
1,99,15,0,222,18,1.3,26.82,13,21.957266
2,31,4,0,0,0,2.6,84.94,-6,0.000000
3,40,18,30,0,0,1.5,89.18,2,0.000000
4,191,1,30,0,0,2.4,84.35,12,0.000000
...,...,...,...,...,...,...,...,...,...
8995,119,11,0,239,16,0.8,24.98,23,23.831336
8996,108,3,30,0,0,5.6,87.78,-1,0.000000
8997,112,7,0,104,525,2.8,47.82,11,26.368066
8998,17,22,0,0,0,3.3,72.11,0,0.000000


In [13]:
df_timesort = df.sort_values(['Day', 'Hour', 'Minute']).reset_index(drop=True)
df_timesort.head(30)

,Day,Hour,Minute,DHI,DNI,WS,RH,T,TARGET
0,0,0,30,0,0,1.5,69.06,-12,0.000000
1,0,1,0,0,0,1.6,71.78,-12,0.000000
2,0,2,0,0,0,1.6,75.20,-12,0.000000
3,0,2,30,0,0,1.5,69.29,-11,0.000000
4,0,3,0,0,0,1.5,72.56,-11,0.000000
5,0,3,30,0,0,1.4,72.55,-11,0.000000
6,0,4,30,0,0,1.3,74.61,-11,0.000000
7,0,5,0,0,0,1.3,73.74,-11,0.000000
8,0,5,30,0,0,1.3,73.73,-11,0.000000
9,0,6,0,0,0,1.4,72.22,-12,0.000000


In [14]:
df['time_idx'] = df['Day'] * 24 * 2 + df['Hour'] * 2 + (df['Minute'] // 30)
df = df.sort_values('time_idx').reset_index(drop=True)
df

,Day,Hour,Minute,DHI,DNI,WS,RH,T,TARGET,time_idx
0,0,0,30,0,0,1.5,69.06,-12,0.000000,1
1,0,1,0,0,0,1.6,71.78,-12,0.000000,2
2,0,2,0,0,0,1.6,75.20,-12,0.000000,4
3,0,2,30,0,0,1.5,69.29,-11,0.000000,5
4,0,3,0,0,0,1.5,72.56,-11,0.000000,6
...,...,...,...,...,...,...,...,...,...,...
8995,208,5,30,26,383,0.8,56.18,13,5.630068,9995
8996,208,6,0,41,578,1.1,47.46,15,13.887196,9996
8997,208,6,30,52,699,1.4,44.51,17,23.269925,9997
8998,208,7,0,61,776,1.7,37.80,19,33.027555,9998


In [15]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 데이터 불러오기
df = pd.read_csv('./study-w10p1/data.csv')

# 2. 시간 순서 정렬
df = df.sort_values(['Day', 'Hour', 'Minute']).reset_index(drop=True)

# 3. 30분 단위 시간 인덱스 생성
df['time_idx'] = df['Day'] * 48 + df['Hour'] * 2 + (df['Minute'] // 30)

# 4. 시간 파생변수
df['hour_float'] = df['Hour'] + df['Minute'] / 60

# 주기형 시간 변수
df['hour_sin'] = np.sin(2 * np.pi * df['hour_float'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_float'] / 24)

# Day도 주기형으로 변환 (1년 365일 가정)
df['day_sin'] = np.sin(2 * np.pi * df['Day'] / 365)
df['day_cos'] = np.cos(2 * np.pi * df['Day'] / 365)

# 5. lag feature
df['target_lag1'] = df['TARGET'].shift(1)     # 30분 전
df['target_lag2'] = df['TARGET'].shift(2)     # 1시간 전
df['target_lag48'] = df['TARGET'].shift(48)   # 1일 전 같은 시각

# 기상 변수 lag
df['DHI_lag1'] = df['DHI'].shift(1)
df['DNI_lag1'] = df['DNI'].shift(1)

# 6. rolling feature
df['target_roll2'] = df['TARGET'].shift(1).rolling(2).mean()    # 최근 1시간 평균
df['target_roll6'] = df['TARGET'].shift(1).rolling(6).mean()    # 최근 3시간 평균
df['target_roll48'] = df['TARGET'].shift(1).rolling(48).mean()  # 최근 1일 평균

# 7. 결측 제거
df = df.dropna().reset_index(drop=True)

# 8. 사용할 입력 변수
features = [
    'Day', 'Hour', 'Minute',
    'DHI', 'DNI', 'WS', 'RH', 'T',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'target_lag1', 'target_lag2', 'target_lag48',
    'DHI_lag1', 'DNI_lag1',
    'target_roll2', 'target_roll6', 'target_roll48'
]

target = 'TARGET'

X = df[features]
y = df[target]

# 9. 시계열 분할 (뒤쪽 20%를 검증용으로)
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
y_train = y.iloc[:split_idx]

X_valid = X.iloc[split_idx:]
y_valid = y.iloc[split_idx:]

# 10. LightGBM 모델
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# 11. 학습
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric='l1'
)

# 12. 예측
pred = model.predict(X_valid)

# 음수 예측 방지
pred = np.clip(pred, 0, None)

# 13. 평가
mae = mean_absolute_error(y_valid, pred)
rmse = np.sqrt(mean_squared_error(y_valid, pred))

print('MAE :', mae)
print('RMSE:', rmse)

# 14. 결과 확인
result = X_valid.copy()
result['actual'] = y_valid.values
result['pred'] = pred

print(result[['actual', 'pred']].head(20))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000160 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3524
[LightGBM] [Info] Number of data points in the train set: 7161, number of used features: 20
[LightGBM] [Info] Start training from score 16.727776
MAE : 0.8126065340653973
RMSE: 1.6064232081729857
         actual       pred
7161   2.439562   2.439921
7162   0.000000   0.103299
7163   0.000000   0.000000
7164   0.000000   0.000000
7165   0.000000   0.000000
7166   0.000000   0.000000
7167   0.000000   0.000000
7168   0.000000   0.000000
7169   0.000000   0.000000
7170   0.000000   0.000000
7171   0.000000   0.000000
7172   0.000000   0.000000
7173   0.000000   0.000000
7174   0.000000   0.000000
7175   0.000000   0.000000
7176   0.000000   0.000000
7177   0.000000   0.000000
7178   3.378041   3.325588
7179  10.227845   9.945074
7180

In [21]:
df.loc[7161]

Day               166.000000
Hour               19.000000
Minute              0.000000
DHI                24.000000
DNI                26.000000
WS                  0.700000
RH                 75.710000
T                  18.000000
TARGET              2.439562
time_idx         8006.000000
hour_float         19.000000
hour_sin           -0.965926
hour_cos            0.258819
day_sin             0.280231
day_cos            -0.959933
target_lag1         6.849465
target_lag2         3.377744
target_lag48        0.656798
DHI_lag1           59.000000
DNI_lag1           84.000000
target_roll2        5.113604
target_roll6       32.884899
target_roll48      20.651242
Name: 7161, dtype: float64

In [19]:
df.loc[7178]

Day               167.000000
Hour                5.000000
Minute              0.000000
DHI                23.000000
DNI               189.000000
WS                  1.900000
RH                 92.800000
T                  13.000000
TARGET              3.378041
time_idx         8026.000000
hour_float          5.000000
hour_sin            0.965926
hour_cos            0.258819
day_sin             0.263665
day_cos            -0.964614
target_lag1         0.000000
target_lag2         0.000000
target_lag48        0.000000
DHI_lag1            0.000000
DNI_lag1            0.000000
target_roll2        0.000000
target_roll6        0.000000
target_roll48      20.272011
Name: 7178, dtype: float64

In [17]:
print(result[['actual', 'pred']].tail(20))

         actual       pred
8932   0.000000   0.000000
8933   0.000000   0.000000
8934   0.000000   0.000000
8935   0.000000   0.000000
8936   0.000000   0.000000
8937   0.000000   0.000000
8938   0.000000   0.000000
8939   0.000000   0.000000
8940   0.000000   0.000000
8941   0.000000   0.000000
8942   0.000000   0.000000
8943   0.000000   0.000000
8944   0.000000   0.000000
8945   0.000000   0.000000
8946   0.000000   0.000000
8947   5.630068   5.375419
8948  13.887196  13.676894
8949  23.269925  21.653553
8950  33.027555  33.230080
8951  42.691399  43.488894


In [18]:
df.iloc[8947]

Day               208.000000
Hour                5.000000
Minute             30.000000
DHI                26.000000
DNI               383.000000
WS                  0.800000
RH                 56.180000
T                  13.000000
TARGET              5.630068
time_idx         9995.000000
hour_float          5.500000
hour_sin            0.991445
hour_cos            0.130526
day_sin            -0.425000
day_cos            -0.905193
target_lag1         0.000000
target_lag2         0.000000
target_lag48        0.000000
DHI_lag1            0.000000
DNI_lag1            0.000000
target_roll2        0.000000
target_roll6        0.000000
target_roll48      20.157199
Name: 8947, dtype: float64